# Regional outbreak probability run

This applies the London Mathsy stochastic forecast to the **eight non-London English regions**. London is excluded because it has a separate observed-data workflow. Each region is conditioned separately on its own latest four synthetic weekly observations, then forecast for six weeks with stochastic S/H/E/I/Q/D paths.

The London-fitted biological, contact and seasonal quantities are transferred to each target region. The fitted external seeding value is London-specific: for region $r$ it is scaled as $s_r=s_{\mathrm L}N_r/N_{\mathrm L}$ and then introduced only into region $r$. London is retained internally in the mobility system and threshold calculation, but is not forecast, plotted, or scored here. Region-specific population, protection, density/citiness, mobility, recent age mix, cases, latent state, and recent transmission correction enter each forecast. Synthetic regional histories are not independent observed surveillance series, so this remains a scenario model rather than regional validation.

In [ ]:
from pathlib import Path
import importlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd()
if REPO.name == 'outbreak_probability_model':
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from outbreak_probability_model.model import load_default_inputs
import outbreak_probability_model.london_calibration as london_calibration
london_calibration = importlib.reload(london_calibration)
import outbreak_probability_model.regional_outbreak_forecast as regional_forecast
regional_forecast = importlib.reload(regional_forecast)
forecast_all_regions = regional_forecast.forecast_all_regions
load_synthetic_regional_history = regional_forecast.load_synthetic_regional_history
case_burden_scaled_thresholds = regional_forecast.case_burden_scaled_thresholds
historical_regional_audit = regional_forecast.historical_regional_audit
find_below_to_above_origins = regional_forecast.find_below_to_above_origins
forecast_region = regional_forecast.forecast_region
DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS = (
    london_calibration.DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS
)
PIPELINE_VERSION = london_calibration.PIPELINE_VERSION
load_london_fitted_parameters = london_calibration.load_london_fitted_parameters

INPUT = REPO / 'experiments' / 'measles_local_age' / 'inputs'
OUTPUT = REPO / 'experiments' / 'measles' / 'all_regions' / 'outbreak_probability'
OUTPUT.mkdir(parents=True, exist_ok=True)
FITTED_PARAMETERS_PATH = DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS
_, FITTED_PARAMETER_VECTOR = load_london_fitted_parameters(FITTED_PARAMETERS_PATH)
print('Loaded pipeline:', PIPELINE_VERSION)
print('Regional transfer vector:', FITTED_PARAMETERS_PATH)
print('Seasonality:', {name: FITTED_PARAMETER_VECTOR[name] for name in
                       ('seasonal_amplitude', 'seasonal_peak_week')})
plt.style.use('seaborn-v0_8-whitegrid')

def format_conditioning(values):
    def one(value):
        value = float(value)
        return str(int(round(value))) if np.isclose(value, round(value)) else f'{value:.1f}'
    return ' → '.join(one(value) for value in values)

## Settings

The outbreak event matches the London notebook: **at least one forecast week is strictly above the region's weekly threshold**. London is fixed at 15. Other thresholds are scaled by population relative to London, with a floor of 3 and a cap equal to the London threshold. Past case burden is not used in the primary threshold; it remains available only as a sensitivity scenario.

The final forecast uses `FORECAST_DT=0.02` days, matching the London model. This is slower than the earlier interactive regional setting but avoids changing numerical resolution between London and the transferred regional forecasts.

In [ ]:
HISTORY_WEEKS = 4
FORECAST_WEEKS = 6
N_SIMULATIONS = 100       # trajectories used for probabilities and intervals
N_DISPLAY_PATHS = 20
LONDON_WEEKLY_THRESHOLD = 15
MINIMUM_REGIONAL_THRESHOLD = 3
CASE_BURDEN_WEIGHT = 0.0  # primary definition: population only
NATIONAL_TREND_WEIGHT = 1.0  # synthetic regional timing comes from national weekly data
FORECAST_DT = 0.02       # same final time step as the London model
RANDOM_SEED = 20260818
EXCLUDED_REGIONS = ('London',)

inputs = load_default_inputs(input_dir=INPUT)
case_history, age_history = load_synthetic_regional_history(INPUT)
thresholds = case_burden_scaled_thresholds(
    inputs, case_history, london_threshold=LONDON_WEEKLY_THRESHOLD,
    minimum_threshold=MINIMUM_REGIONAL_THRESHOLD,
    case_burden_weight=CASE_BURDEN_WEIGHT,
)
thresholds.drop(labels=list(EXCLUDED_REGIONS)).to_frame()

## How the synthetic regional series is generated

For each region, age group, and reporting year/period, its observed annual total is distributed over weeks using the national England weekly pattern:

$$\text{synthetic}_{r,a,w}=\text{England weekly}_{w}\times\frac{\text{annual regional cases}_{r,a}}{\sum_{v\in\text{period}}\text{England weekly}_{v}}.$$

Thus every region retains its own annual case burden, while within-year timing comes from the national weekly data. London rows may exist in the shared prepared input, but this notebook filters them from forecasting and validation.

## The four observations used at each origin

Only these four trailing values are passed to each region's local history-conditioning step. The final value is week 0; forecast week 1 is the following Monday.

In [ ]:
recent_four = (case_history.groupby('region', group_keys=False).tail(HISTORY_WEEKS)
               .assign(relative_week=lambda x: x.groupby('region').cumcount() - HISTORY_WEEKS + 1)
               .pivot(index='region', columns='relative_week', values='observed_cases'))
recent_four.columns = [f'week_{x}' for x in recent_four.columns]
recent_four.drop(index=list(EXCLUDED_REGIONS), errors='ignore').round(2)

## Run eight independently conditioned stochastic forecasts

This is computationally heavier than the previous allocation model because every line is an actual compartmental simulation.

In [ ]:
results = forecast_all_regions(
    input_dir=INPUT, inputs=inputs, fitted_parameters_path=FITTED_PARAMETERS_PATH,
    london_threshold=LONDON_WEEKLY_THRESHOLD,
    minimum_threshold=MINIMUM_REGIONAL_THRESHOLD,
    history_weeks=HISTORY_WEEKS, horizon_weeks=FORECAST_WEEKS,
    n_simulations=N_SIMULATIONS, random_seed=RANDOM_SEED,
    forecast_dt=FORECAST_DT, conditioning_maxiter=30,
    case_burden_weight=CASE_BURDEN_WEIGHT,
    exclude_regions=EXCLUDED_REGIONS,
    national_trend_weight=NATIONAL_TREND_WEIGHT,
)
print(f'Completed {len(results)} regional forecasts.')

## Actual stochastic paths

Thin red lines are individual simulated futures, blue is the median, shading is p10–p90, the black point is the observed/synthetic week-0 anchor, and the dashed line is that region's weekly threshold. Unlike the old plot, these are not national totals reallocated after simulation.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True)
for ax, (region, result) in zip(axes.flat, results.items()):
    paths = result.trajectories
    shown_ids = sorted(paths.simulation.unique())[:N_DISPLAY_PATHS]
    shown = paths.loc[paths.simulation.isin(shown_ids)]
    for _, path in shown.groupby('simulation'):
        x = np.r_[0, path.week.to_numpy()]
        y = np.r_[result.current_cases, path.weekly_cases.to_numpy()]
        ax.plot(x, y, color='tab:red', alpha=.20, lw=.8)
    s = result.weekly_summary
    x = np.r_[0, s.week.to_numpy()]
    median = np.r_[result.current_cases, s.median_cases.to_numpy()]
    lo = np.r_[result.current_cases, s.p10_cases.to_numpy()]
    hi = np.r_[result.current_cases, s.p90_cases.to_numpy()]
    ax.fill_between(x, lo, hi, color='tab:blue', alpha=.18)
    ax.plot(x, median, color='tab:blue', lw=2)
    ax.scatter([0], [result.current_cases], color='black', s=25, zorder=5)
    ax.axhline(result.outbreak_threshold, color='black', ls='--', lw=1.3)
    ax.set_title(f'{region}: P(outbreak)={result.outbreak_probability:.0%}')
    history_text = format_conditioning(recent_four.loc[region].to_numpy())
    ax.text(.02, .96, f'conditioning: {history_text}  (last = week 0)',
            transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(facecolor='white', alpha=.78, edgecolor='none', pad=2))
    ax.set_xticks(range(FORECAST_WEEKS + 1))
    ax.set_ylabel('Weekly cases')
for ax in axes.flat[len(results):]: ax.set_visible(False)
for ax in axes[-1]: ax.set_xlabel('Forecast week (0 = origin)')
fig.suptitle('Four-week-conditioned Mathsy SDE paths by region', y=1.01, fontsize=15)
fig.tight_layout()
fig.savefig(OUTPUT / 'regional_stochastic_paths.png', dpi=180, bbox_inches='tight')
plt.show()

## Outbreak probability summary and saved paths

In [ ]:
summary_rows = []
for region, result in results.items():
    flags = result.trajectories.groupby('simulation').crosses_outbreak_threshold.first()
    p = flags.mean(); se = np.sqrt(p * (1 - p) / len(flags))
    summary_rows.append({
        'region': region, 'origin_date': result.origin_date,
        'four_week_history': recent_four.loc[region].round(2).tolist(),
        'current_cases': result.current_cases,
        'weekly_outbreak_threshold': result.outbreak_threshold,
        'outbreak_probability': p,
        'mc_95_low': max(0, p - 1.96 * se), 'mc_95_high': min(1, p + 1.96 * se),
    })
summary = pd.DataFrame(summary_rows).sort_values('outbreak_probability', ascending=False)
all_paths = pd.concat([r.trajectories for r in results.values()], ignore_index=True)
all_weekly = pd.concat([r.weekly_summary for r in results.values()], ignore_index=True)
conditioning = pd.DataFrame([r.conditioning_diagnostics for r in results.values()])
conditioning_fit = pd.concat([r.conditioning_fit for r in results.values()], ignore_index=True)
summary.to_csv(OUTPUT / 'regional_outbreak_probabilities.csv', index=False)
all_paths.to_csv(OUTPUT / 'regional_stochastic_paths.csv', index=False)
all_weekly.to_csv(OUTPUT / 'regional_weekly_summary.csv', index=False)
conditioning.to_csv(OUTPUT / 'regional_history_conditioning_diagnostics.csv', index=False)
conditioning_fit.to_csv(OUTPUT / 'regional_history_conditioning_fit.csv', index=False)
summary.style.format({'outbreak_probability': '{:.1%}', 'mc_95_low': '{:.1%}', 'mc_95_high': '{:.1%}'})

## Check the four-week conditioning

The multiplier is constrained to 0.8–1.25 so four synthetic values cannot replace the shared long-term calibration. Inspect optimizer success and residuals before trusting a regional forecast.

In [ ]:
conditioning[['region', 'recent_transmission_multiplier', 'national_recent_log_slope',
              'raw_model_log_slope', 'trend_correction_week_1', 'trend_correction_week_6',
              'origin_exposed_total',
              'origin_infectious_total', 'origin_sick_total',
              'conditioning_objective', 'optimizer_success']].round(3)

## Historical synthetic-data comparison

This repeats the leakage-safe London evaluation for every region. At each historical origin, only the four observations ending at week 0 are available to conditioning. The six later synthetic observations are withheld until after simulation and then overlaid in black.

The default selects four origins spread across the available history. Increase simulations for final estimates. A complete every-week audit is possible but computationally expensive because it refits the latent origin state for every region–origin pair.

In [ ]:
HISTORICAL_ORIGINS = 6          # origins spread across the available history
HISTORICAL_SIMULATIONS = 100    # trajectories used for probabilities and intervals
HISTORICAL_DISPLAY_PATHS = 20   # representative trajectories drawn in each panel

audit = historical_regional_audit(
    input_dir=INPUT, inputs=inputs, n_origins=HISTORICAL_ORIGINS,
    fitted_parameters_path=FITTED_PARAMETERS_PATH,
    london_threshold=LONDON_WEEKLY_THRESHOLD,
    minimum_threshold=MINIMUM_REGIONAL_THRESHOLD,
    history_weeks=HISTORY_WEEKS, horizon_weeks=FORECAST_WEEKS,
    n_simulations=HISTORICAL_SIMULATIONS, random_seed=RANDOM_SEED + 50000,
    forecast_dt=FORECAST_DT, conditioning_maxiter=30,
    case_burden_weight=CASE_BURDEN_WEIGHT,
    exclude_regions=EXCLUDED_REGIONS,
    national_trend_weight=NATIONAL_TREND_WEIGHT,
)
audit.regional_metrics.round(3)

In [ ]:
historical_plot_dir = OUTPUT / 'historical_origin_plots'
historical_plot_dir.mkdir(parents=True, exist_ok=True)
selected_origins = sorted(audit.origin_metrics.origin_date.unique())
for origin in selected_origins:
    origin = pd.Timestamp(origin)
    fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True)
    audit_regions = list(results)
    for ax, region in zip(axes.flat, audit_regions):
        result = audit.forecasts[(region, origin)]
        paths = result.trajectories
        shown_ids = sorted(paths.simulation.unique())[:HISTORICAL_DISPLAY_PATHS]
        for _, path in paths.loc[paths.simulation.isin(shown_ids)].groupby('simulation'):
            ax.plot(np.r_[0, path.week], np.r_[result.current_cases, path.weekly_cases],
                    color='tab:red', alpha=.16, lw=.7)
        comparison = audit.weekly_comparison.query('region == @region and origin_date == @origin')
        s = result.weekly_summary
        x = np.arange(FORECAST_WEEKS + 1)
        ax.fill_between(x, np.r_[result.current_cases, s.p10_cases],
                        np.r_[result.current_cases, s.p90_cases], color='tab:blue', alpha=.16)
        ax.plot(x, np.r_[result.current_cases, s.median_cases], color='tab:blue', lw=2, label='median')
        ax.plot(x, np.r_[result.current_cases, comparison.observed_cases],
                color='black', marker='o', lw=2, label='withheld synthetic truth')
        ax.axhline(result.outbreak_threshold, color='black', ls='--', lw=1.1)
        metric = audit.origin_metrics.query('region == @region and origin_date == @origin').iloc[0]
        ax.set_title(f"{region}: P={metric.predicted_probability:.0%}, observed={bool(metric.observed_outbreak)}")
        conditioning_values = (case_history.query('region == @region and date <= @origin')
                               .sort_values('date').tail(HISTORY_WEEKS).observed_cases)
        history_text = format_conditioning(conditioning_values)
        ax.text(.02, .96, f'conditioning: {history_text}  (last = week 0)',
                transform=ax.transAxes, va='top', fontsize=8,
                bbox=dict(facecolor='white', alpha=.78, edgecolor='none', pad=2))
        ax.set_ylabel('Weekly cases'); ax.set_xticks(x)
    for ax in axes.flat[len(audit_regions):]: ax.set_visible(False)
    axes.flat[0].legend(fontsize=7)
    for ax in axes[-1]: ax.set_xlabel('Week (0 = forecast origin)')
    fig.suptitle(f'Historical origin {origin.date()}: four-week conditioning, six-week holdout', y=1.01)
    fig.tight_layout()
    fig.savefig(historical_plot_dir / f'origin_{origin:%Y%m%d}.png', dpi=160, bbox_inches='tight')
    plt.show()

In [ ]:
audit.weekly_comparison.to_csv(OUTPUT / 'historical_weekly_comparison.csv', index=False)
audit.origin_metrics.to_csv(OUTPUT / 'historical_origin_metrics.csv', index=False)
audit.regional_metrics.to_csv(OUTPUT / 'historical_regional_metrics.csv', index=False)

overall_metrics = pd.Series({
    'region_origin_forecasts': len(audit.origin_metrics),
    'mean_brier_score': audit.origin_metrics.brier_score.mean(),
    'mean_absolute_error': audit.origin_metrics.mae.mean(),
    'empirical_80_interval_coverage': audit.weekly_comparison.covered_80.mean(),
    'observed_outbreak_rate': audit.origin_metrics.observed_outbreak.mean(),
    'mean_predicted_probability': audit.origin_metrics.predicted_probability.mean(),
})
overall_metrics.round(3)

## Targeted rolling audit: starts below threshold, crosses later

This scans **every eligible historical origin** for the event you care about: week 0 is at or below its past-only regional threshold, but at least one of the six withheld weeks is above it. Future data are used only to label audit examples; each stochastic forecast is still fitted using observations available through week 0 only.

To avoid hundreds of repeated compartment fits, the expensive model is run for one representative episode per affected region and crossing lead time (weeks 1–6). The complete candidate table remains available for extending the audit.

In [ ]:
crossing_candidates = find_below_to_above_origins(
    input_dir=INPUT, inputs=inputs,
    london_threshold=LONDON_WEEKLY_THRESHOLD,
    minimum_threshold=MINIMUM_REGIONAL_THRESHOLD,
    case_burden_weight=CASE_BURDEN_WEIGHT,
    history_weeks=HISTORY_WEEKS, horizon_weeks=FORECAST_WEEKS,
    exclude_regions=EXCLUDED_REGIONS,
)
crossing_candidates.to_csv(OUTPUT / 'all_below_to_above_candidates.csv', index=False)
print(f'{len(crossing_candidates)} below-to-above episodes across '      f'{crossing_candidates.region.nunique()} regions')
if crossing_candidates.empty:
    print('No qualifying below-to-above episodes were found at the current thresholds; the targeted audit will be skipped.')
    display(crossing_candidates)
else:
    display(crossing_candidates.groupby(['region', 'first_crossing_week']).size().unstack(fill_value=0))

In [ ]:
CROSSING_AUDIT_SIMULATIONS = 20  # increase to 500+ for final probabilities
selected_crossings = (crossing_candidates
    .sort_values(['region', 'first_crossing_week', 'origin_date'])
    .groupby(['region', 'first_crossing_week'], as_index=False).first())
crossing_forecasts = {}
crossing_rows = []
for row_number, row in selected_crossings.iterrows():
    result = forecast_region(
        row.region, case_history, age_history, inputs=inputs,
        outbreak_threshold=row.outbreak_threshold, origin_date=row.origin_date,
        fitted_parameters_path=FITTED_PARAMETERS_PATH,
        history_weeks=HISTORY_WEEKS, horizon_weeks=FORECAST_WEEKS,
        n_simulations=CROSSING_AUDIT_SIMULATIONS,
        random_seed=RANDOM_SEED + 90000 + row_number * 1009,
        forecast_dt=FORECAST_DT, conditioning_maxiter=8,
        national_trend_weight=NATIONAL_TREND_WEIGHT,
    )
    key = (row.region, pd.Timestamp(row.origin_date))
    crossing_forecasts[key] = result
    probability = result.trajectories.groupby('simulation').crosses_outbreak_threshold.first().mean()
    crossing_rows.append({**row.to_dict(), 'predicted_probability': probability})
crossing_results = pd.DataFrame.from_records(
    crossing_rows, columns=[*selected_crossings.columns, 'predicted_probability']
)
crossing_results.to_csv(OUTPUT / 'selected_below_to_above_forecasts.csv', index=False)
if crossing_results.empty:
    print('Targeted below-to-above forecasts skipped because the candidate scan was empty.')
crossing_results[['region', 'origin_date', 'origin_cases', 'outbreak_threshold',
                  'first_crossing_week', 'predicted_probability']]

In [ ]:
crossing_plot_dir = OUTPUT / 'below_to_above_plots'
crossing_plot_dir.mkdir(parents=True, exist_ok=True)
for region, region_rows in crossing_results.groupby('region'):
    region_rows = region_rows.sort_values('first_crossing_week')
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
    for ax, (_, row) in zip(axes.flat, region_rows.iterrows()):
        origin = pd.Timestamp(row.origin_date)
        result = crossing_forecasts[(region, origin)]
        shown = result.trajectories[result.trajectories.simulation <= N_DISPLAY_PATHS]
        for _, path in shown.groupby('simulation'):
            ax.plot(np.r_[0, path.week], np.r_[result.current_cases, path.weekly_cases],
                    color='tab:red', alpha=.16, lw=.7)
        future = np.asarray(row.future_cases, dtype=float)
        s = result.weekly_summary
        x = np.arange(FORECAST_WEEKS + 1)
        ax.fill_between(x, np.r_[result.current_cases, s.p10_cases],
                        np.r_[result.current_cases, s.p90_cases], color='tab:blue', alpha=.18)
        ax.plot(x, np.r_[result.current_cases, s.median_cases], color='tab:blue', lw=2)
        ax.plot(x, np.r_[result.current_cases, future], color='black', marker='o', lw=2)
        ax.axhline(row.outbreak_threshold, color='black', ls='--', lw=1.2)
        conditioning_values = (case_history.query('region == @region and date <= @origin')
                               .sort_values('date').tail(HISTORY_WEEKS).observed_cases)
        ax.text(.02, .96, format_conditioning(conditioning_values), transform=ax.transAxes,
                va='top', fontsize=8, bbox=dict(facecolor='white', alpha=.75, edgecolor='none'))
        ax.set_title(f"origin {origin.date()}, crosses week {int(row.first_crossing_week)}; "
                     f"P={row.predicted_probability:.0%}")
        ax.set_ylabel('Weekly cases'); ax.set_xticks(x)
    for ax in axes.flat[len(region_rows):]: ax.set_visible(False)
    for ax in axes[-1]: ax.set_xlabel('Week (0 = origin)')
    fig.suptitle(f'{region}: observed below-to-above threshold episodes', y=1.01)
    fig.tight_layout()
    fig.savefig(crossing_plot_dir / f"{region.lower().replace(' ', '_')}.png",
                dpi=170, bbox_inches='tight')
    plt.show()

## Interpretation

- Probability means the fraction of stochastic paths with **any one of weeks 1–6 strictly above the regional weekly threshold**.
- Each region uses exactly four trailing synthetic observations and no future cases.
- Paths differ because of compartmental Wiener noise, discrete future importations, and optional Poisson observation sampling.
- The regional histories inherit national timing by construction. They cannot validate genuinely region-specific dynamics.
- The current-origin forecasts use 1,000 simulations and `FORECAST_DT=0.02`, matching the final London settings. Perform independent rolling-origin validation for each region once real weekly regional surveillance data are available.